In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import torch

from src.config import SEED, BATCH_SIZE, LR, WEIGHT_DECAY, NUM_EPOCHS, PATIENCE
from src.cnn import CNN, train_cnn

torch.manual_seed(SEED)

# CNN entrenada desde cero
## Fase 3 — Entrenamiento CNN

A diferencia del MLP, la CNN trabaja directamente sobre los píxeles de la imagen sin necesidad de PCA. Las capas convolucionales detectan patrones espaciales (bordes, texturas, estructuras) que el MLP no puede capturar porque trata a cada píxel como una feature independiente.

**Por qué la CNN puede trabajar con píxeles directamente:**
- Los filtros convolucionales **comparten pesos**: un filtro de 3×3 se aplica en toda la imagen en lugar de tener un peso por píxel
- Cada MaxPool reduce el mapa espacial a la mitad, comprimiendo la información progresivamente
- BatchNorm estabiliza el entrenamiento normalizando las activaciones de cada capa

**Arquitectura: 4 bloques convolucionales**

| Bloque | Filtros | Salida espacial       |
|--------|---------|-----------------------|
| Conv1  | 32      | 224×224 → 112×112     |
| Conv2  | 64      | 112×112 → 56×56       |
| Conv3  | 128     | 56×56 → 28×28         |
| Conv4  | 256     | 28×28 → 14×14         |

La cabeza clasificadora aplana el mapa 14×14×256 a un vector de 50 176 features y usa dos capas densas (512 → 1).

Pipeline:
1. Inspeccionar la arquitectura y contar parámetros
2. Entrenar la CNN sobre las imágenes preprocesadas

## 1. Inspección de la arquitectura

Instanciamos el modelo para ver su estructura y el total de parámetros entrenables.

In [ ]:
model = CNN()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {total_params:,}")
print()
print(model)

## 2. Entrenamiento

Entrenamos la CNN sobre las imágenes preprocesadas en `data/processed/`. Recibe las imágenes directamente, sin reducción de dimensionalidad previa.

- **BCEWithLogitsLoss**: pérdida binaria con sigmoid integrado, más estable numéricamente que Sigmoid + BCELoss
- **Adam**: lr=1e-3, weight_decay=1e-4
- **Early stopping**: frena si `val_loss` no mejora en 10 epochs consecutivos, guardando el mejor checkpoint
- **Data augmentation** (solo en train): flip horizontal, rotación ±10°, brillo y contraste aleatorio

In [ ]:
historial = train_cnn()